# Exercise 2: Locating CARE evidence in release documentation

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Endangered Languages and Ethics · version 0.1.0*

> Review a published project against CARE. Separate documented decisions, missing reporting and your own recommendations; do not infer absent consent from silence in a paper.

**Setup.** `pip install requests huggingface_hub fasttext-wheel transformers torch sentencepiece "turkicnlp[hfst]==0.5.0"`; each notebook uses only part of this, and Exercise 1 also calls `git`. Data come from commit- or tag-pinned GitHub files and are hash-checked: the UD 2.17 TueCL treebanks, the KazNERD and KyrgyzNER releases and their README files, and the TIL corpus README. The audit in Exercise 1 also reads the book's UD 2.17 inventory (`tables/tbl-ud217-inventory/data.csv`), the NLLB-200 tokeniser (`NLLB_PATH`) and GlotLID (`GLOTLID_PATH`), and queries live services, so its live columns are dated observations, not pinned results. Run the notebooks from the `exercises/` folder: the first code cell imports the chapter's `observation` and `hub_count` helpers from `../code/remaining/resources.py`. Downloads are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`).

The cell searches the README files of KazNERD and KyrgyzNER (both at pinned commits) for lines that bear on six CARE questions and prints them with line numbers. Code blocks are skipped. The code prints *not documented in this file* when its patterns find no match. Treat that as a search result until you have read the full file; it can miss relevant wording.

In [ ]:
import collections, csv, datetime, difflib, hashlib, json, os, re, statistics, subprocess, sys, time
from pathlib import Path
import requests
sys.path.insert(0, str(Path("../code/remaining").resolve()))
from resources import observation, hub_count

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/"
KYRGYZNER = "https://raw.githubusercontent.com/Akyl-AI/KyrgyzNER/b76b9c9c9c4f5d681c25308cc0d8dd9f0bd6569c/"
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
UD = "https://raw.githubusercontent.com/UniversalDependencies/{}/r2.17/{}"
SHA256 = {
    KAZNERD + "README.md": "2d56859aa35eadafb8bc560b375d5128627e3bab3517bc74377c7818905e3614",
    KAZNERD + "KazNERD/IOB2_train.txt": "e5a80f6d4bbff499a4bf7cbf96a169feb761411151fb45abebeaae06995ff647",
    KAZNERD + "KazNERD/IOB2_test.txt": "359a1f61f6a21280c6c47b4fde9a5adcd07411d62015aa6e32d940c51f163b7a",
    KYRGYZNER + "README.md": "c5beec0dd94156f5cdae4c5f7ffafeb23704f6fd1024ed4cb505cc8ef498a46c",
    TIL + "til_corpus/README.md": "da88f81cc7a23fb46425dcf236f7c89ad0f63663d2141d8fb7165ea8015308d8",
    UD.format("UD_Turkish-TueCL", "tr_tuecl-ud-test.conllu"): "2508f47e57b9b6a1d7241d0ab4a918d84c5cbb0455bbd961ac5231e31a1d311e",
    UD.format("UD_Azerbaijani-TueCL", "az_tuecl-ud-test.conllu"): "8b315f8b065f094e38482465706ba44a9caf1b082fc8bbc94908f6fdeee8c3b5",
    UD.format("UD_Uzbek-TueCL", "uz_tuecl-ud-test.conllu"): "23257711196df0bbfaf5b25397d7c6102fbc8800ed83b15eb77342929ee0e469",
    UD.format("UD_Kyrgyz-TueCL", "ky_tuecl-ud-test.conllu"): "08224be6b74346f582234f54527412dd6265c8d8b42e7d6ef463aade85bdcd9b",
}
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256(path.read_bytes()).hexdigest(); FETCHED[url] = h
    if url in SHA256 and h != SHA256[url]:
        raise ValueError(f"{url} differs from the pinned file")
    return path

def read_conll(path):
    """KazNERD-style column file -> [(words, tags)]."""
    sents, words, tags = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines() + [""]:
        if not line.strip():
            if words: sents.append((words, tags))
            words, tags = [], []
        elif not line.startswith("# "):
            c = line.split("\t") if "\t" in line else line.split()
            words.append(c[0]); tags.append(c[-1].strip())
    return sents

def show(obs):
    """One audit cell: the value, or the reason why there is none. Zero, not found, error and not checked stay distinct."""
    return {"observed": lambda o: str(o["value"]), "not_found": lambda o: "not found", "error": lambda o: "ERROR",
            "not_checked": lambda o: "not checked"}[obs["status"]](obs)
print("helpers loaded from code/remaining/resources.py:", observation.__name__, hub_count.__name__)

In [ ]:
# Evidence locator for a CARE review: where does the release documentation speak to each question?
# It finds candidate lines; the reviewer reads them and decides. A question with no line is "not documented here".
QUESTIONS = {
    "C: who benefits, and how was the use case chosen?": r"benefit|need for|aim|community|advance",
    "A: who decided on collection and release?": r"permission|approv|governance|decid|agreement",
    "A: source rights and licence of the text": r"licen[cs]e|copyright|CC BY|terms of use|extracted from|news portal|television",
    "R: who annotated, how were they trained and paid?": r"annotated by|annotators?\b|linguists?\b|volunteers?\b|students?\b|paid\b|compensat",
    "R: maintenance and contact for corrections": r"\bissues?\b|pull request|contact|maintain|changelog|e-?mail",
    "E: personal data, consent and withdrawal": r"consent|personal|privacy|anonymi|withdraw|remov",
}
DOCS = {"KazNERD README": KAZNERD + "README.md", "KyrgyzNER README": KYRGYZNER + "README.md"}
for doc, url in DOCS.items():
    lines, fenced = [], False
    for l in fetch(url).read_text(encoding="utf-8").splitlines():      # skip code blocks: they document software, not data
        if l.strip().startswith("```"): fenced = not fenced; continue
        if l.strip() and not fenced: lines.append(l.strip())
    print(f"\n=== {doc} ({len(lines)} non-empty lines)")
    for q, pattern in QUESTIONS.items():
        hits = [(i + 1, l) for i, l in enumerate(lines) if re.search(pattern, l, re.I) and not l.startswith(("@", "title", "author", "booktitle", "url", "doi", "pages", "publisher", "address", "month", "year", "abstract"))]
        print(f"- {q}")
        if not hits:
            print("    not documented in this file")
        for n, l in hits[:3]:
            print(f"    line {n}: {l[:110]}{'…' if len(l) > 110 else ''}")
        if len(hits) > 3:
            print(f"    (+{len(hits) - 3} more lines)")

## Worked solution

Output of `experiments/ch17_exercises/run_ch17_exercises.py`:

```

=== KazNERD README (38 non-empty lines)
- C: who benefits, and how was the use case chosen?
    not documented in this file
- A: who decided on collection and release?
    not documented in this file
- A: source rights and licence of the text
    line 11: KazNERD contains 112,702 sentences, extracted from the television news text, and 136,333 annotations for 25 en…
- R: who annotated, how were they trained and paid?
    line 12: All sentences in the dataset were manually annotated by two native Kazakh-speaking linguists, supervised by an…
- R: maintenance and contact for corrections
    not documented in this file
- E: personal data, consent and withdrawal
    not documented in this file

=== KyrgyzNER README (59 non-empty lines)
- C: who benefits, and how was the use case chosen?
    line 7: This project provides both the dataset and baseline models, aiming to advance NLP research for low-resource Tu…
    line 50: We welcome contributions from the **NLP community**. Please open issues or pull requests if you’d like to impr…
- A: who decided on collection and release?
    line 18: - High-quality dataset with **κ = 0.89 inter-annotator agreement**
- A: source rights and licence of the text
    line 6: It consists of **1,499 news articles** (10,900 sentences, 140k tokens) from the [24.kg](https://24.kg) news po…
    line 55: ## License
    line 56: - Dataset: **CC BY-NC-SA 4.0**
    (+1 more lines)
- R: who annotated, how were they trained and paid?
    line 16: - Annotated by 59 trained Kyrgyz linguists and students
    line 18: - High-quality dataset with **κ = 0.89 inter-annotator agreement**
    line 47: - **59 volunteers** (mainly students of KSTU) who annotated the dataset
    (+1 more lines)
- R: maintenance and contact for corrections
    line 50: We welcome contributions from the **NLP community**. Please open issues or pull requests if you’d like to impr…
- E: personal data, consent and withdrawal
    not documented in this file
```

**Keyword hits are pointers to evidence.** The pinned READMEs describe KazNERD's television-news source and linguistic annotation, and KyrgyzNER's news source, volunteers and release licence. Read the full passages and the cited papers before recording a finding.

A missing keyword hit does not establish that a topic is absent even from that file: the search can miss different wording. Replace the printed “not documented” result with “no matching passage found by this search” in your review until you have read the document. Likewise, “inter-annotator agreement” is about annotation quality, not agreement to a project's governance.

Separate documented decisions, reporting gaps and recommendations. Do not infer that consent was absent or contributors were unpaid merely because a README does not discuss them. A stated dataset licence also does not document every source right or participant permission.

## Your turn — extension

Run the locator on the paper of one of the two datasets (paste its text into a file) and on a third release of your choice. Write a one-page CARE review in three columns: documented (with line or page), not documented, recommendation.